In [1]:
import mlflow
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
import numpy as np
from itertools import product
import matplotlib.pyplot as plt
from sklearn.model_selection import TimeSeriesSplit
from collections import defaultdict

import sys
sys.path.insert(0, "../../run")
from const import REPO_PATH
from experiment_config import TRAINGING_CONFIG

sys.path.insert(1, f"{REPO_PATH}")
from src.model.experiment_utils import *
from src.model.model_utils import *
from src.model.models import LSTMModel

In [2]:
features_path = f"{TRAINGING_CONFIG['features_path']}/all_combined_features.csv"
output_dir = f"{REPO_PATH}/models/ml_models"

In [3]:
seasons=sorted(TRAINGING_CONFIG['seasons'])
target_dfs=[pd.read_csv(f"{TRAINGING_CONFIG['processed_data_path']}/{season}/all_target_df.csv") for season in seasons[1:]]
for df in target_dfs:
    df['date']= pd.to_datetime(df['date'])
target_df = pd.concat(target_dfs, ignore_index=True)
target_columns=target_df.drop(columns=TRAINGING_CONFIG['key_columns']).columns.tolist()

In [4]:
feature_df = pd.read_csv(features_path)
feature_df['date']= pd.to_datetime(feature_df['date'])
feature_df, target_df=align_on_keys(feature_df, target_df, TRAINGING_CONFIG['key_columns'])

In [5]:
def prepare_temporal_data(feature_df, target_df, sequence_length, target_columns):
    feature_df['date'] = pd.to_datetime(feature_df['date'])
    feature_df['season'] = feature_df['date'].apply(lambda d: d.year if d.month >= 7 else d.year - 1)

    def melt_matches(df):
        rows = []
        for _, row in df.iterrows():
            for is_home in [True, False]:
                team = row['home'] if is_home else row['away']
                opponent = row['away'] if is_home else row['home']
                new_row = row.copy()
                new_row['team'] = team
                new_row['opponent'] = opponent
                new_row['is_home'] = is_home
                rows.append(new_row)
        return pd.DataFrame(rows)

    long_df = melt_matches(feature_df)
    long_df = long_df.sort_values(['team', 'season', 'date'])

    exclude_cols = ['home', 'away', 'date', 'season', 'team', 'opponent', 'is_home']
    feature_cols = [c for c in long_df.columns if c not in exclude_cols]

    sequences = []
    targets = {col: [] for col in target_columns}
    metadata = []

    grouped = long_df.groupby(['team', 'season'])
    for _, group in grouped:
        group = group.sort_values('date')
        values = group[feature_cols].values
        for i in range(sequence_length, len(group)):
            seq = values[i-sequence_length:i]
            target_row = group.iloc[i]
            match_index = target_row.name
            if match_index in target_df.index:
                sequences.append(seq)
                for col in target_columns:
                    targets[col].append(target_df.loc[match_index, col])
                metadata.append({
                    'team': target_row['team'],
                    'date': target_row['date'],
                    'match_index': match_index
                })

    targets_df = pd.DataFrame(targets)
    return np.array(sequences), targets_df, metadata


In [6]:
sequence_length = 5
target_columns = TRAINGING_CONFIG['target_ranges'].keys()  # replace with actual column name
X_seq, y_seq, meta = prepare_temporal_data(feature_df, target_df, sequence_length, target_columns)

In [14]:
X_seq.shape

(2620, 5, 2626)

In [11]:
run_multiclass_distribution_experiment(
    feature_df=X_seq,
    target_df=y_seq,
    target_ranges=TRAINGING_CONFIG['target_ranges'],
    model_wrapper_class=lambda **params: TorchSequenceWrapper(LSTMModel, **params),
    model_param_grid={
        'model_params': [{
            'hidden_dim': 64,
            'num_layers': 2,
            'dropout': 0.3
        }],
        'learning_rate': [0.001],
        'batch_size': [32],
        'epochs': [10],
        'device': ['auto']
    },
    test_size=0.2,
    k=5,
    experiment_name=TRAINGING_CONFIG['experiment_name'],
    uri=f'{REPO_PATH}/mlflow',
    model_name='LSTM',
    key_columns=TRAINGING_CONFIG['key_columns'],
)

IndexError: only integers, slices (`:`), ellipsis (`...`), numpy.newaxis (`None`) and integer or boolean arrays are valid indices